# Polymer-protein binding: A2 on mNET

Build a small polymer (`A2`, a dimer of building block `A`), parametrise it, and optimise
its binding pose on a protein surface (`mNET`).

The binding site comes from a P2Rank pocket prediction shipped in `input/pockets/`
(P2Rank 2.3.1; re-create it with `prank predict -f input/mNET.pdb`).

Runtime: about 2 minutes.

## Setup
Checks that PolyLego, GROMACS and AmberTools are available and pins the thread counts.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "common")))
from polylego_examples import *

# Pin threads for every step launched from this notebook (see examples/README.md).
os.environ.setdefault("OMP_NUM_THREADS", "8")
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

GMX = check_env()
os.environ["POLYLEGO_GMX"] = GMX
# GPU for the GROMACS runs: "auto" (use a compatible GPU if there is one, else CPU),
# "none" (CPU only), or a GPU id such as "0".
GPU = os.environ.get("POLYLEGO_GPU", "auto")
os.makedirs("output", exist_ok=True)

PolyLego   1.4.0  ($CONDA_PREFIX/lib/python3.10/site-packages/PolyLego)
Python     3.10.13
GROMACS    $CONDA_PREFIX/bin/gmx
AmberTools $CONDA_PREFIX/bin/antechamber
OMP_NUM_THREADS        8
OPENBLAS_NUM_THREADS   1
MKL_NUM_THREADS        1


## 1. Inputs

In [2]:
print(open("input/A2.txt").read())
show_mol("input/A.sdf")

## [ BUILDING BLOCK ]
name	type	head_smart	head_bonders	head_deletions	end_smart	end_bonder	end_deletor
A	2	[C][O][H] 	0	1,2	[C][O][H]	1	2

## [ CONNECTION ]
name	donate_mol	accept_mol	repeat	reaction_times
A2	A	A	2	1




[3D view - run the notebook in Jupyter to display it]


Predicted pockets on mNET (top 3 shown in orange; the binding optimisation uses them as starting sites):

In [3]:
pockets = [pocket_center("input/pockets/mNET.pdb_predictions.csv", r) for r in (1, 2, 3)]
for i, p in enumerate(pockets, 1):
    print(f"pocket {i}: " + "  ".join(f"{v:8.3f}" for v in p))
show_complex("input/mNET.pdb", pockets=pockets, focus_ligand=False)

pocket 1:   47.966    48.284    65.119
pocket 2:   20.069    50.662    54.697
pocket 3:   37.422    39.036    45.851


[3D view - run the notebook in Jupyter to display it]


## 2. Assemble and parametrise the polymer

In [4]:
run(["PolyLego-BuildPolymer", "-i", "input", "-o", "output",
     "-t", "input/A2.txt", "-l", "A2"], log="output/build.log")
run(["PolyLego-AssignForceField", "-l", "input", "-t", "input/A2.txt",
     "-o", "output", "-n", "A2", "--gpu", GPU], log="output/assign.log")
show_mol("output/assembly.mol2")

$ PolyLego-BuildPolymer -i input -o output -t input/A2.txt -l A2


  PolyLego 1.4.0 | BuildPolymer
  assembling A2 from input/A2.txt
  Finished in 0 s
    output/A2.sdf
    output/A2_graph.gml
  -> exit 0, 2 s
$ PolyLego-AssignForceField -l input -t input/A2.txt -o output -n A2 --gpu none


  PolyLego 1.4.0 | AssignForceField
  force field for A2 (polymer)
    building block A: GAFF-2.11 + AM1-BCC charges
    junction parameters from capped fragments
    assembly topology
    vacuum energy minimisation + short NVT (GROMACS)
  Finished in 1 min 30 s
    output/assembly.top
    output/assembly.gro
    output/assembly.pdb
  -> exit 0, 93 s


[3D view - run the notebook in Jupyter to display it]


## 3. Optimise the binding pose
`PolyLego-OptimizeBinding --polymers` places the polymer at each predicted pocket and optimises its pose; one result per pocket is written to `output/docking/_<k>/`.

In [5]:
import shutil
shutil.copy("input/mNET.pdb", "output/mNET.pdb")
run(["PolyLego-OptimizeBinding", "--polymers", "--input_path", "./",
     "--ligand_pdb", "assembly.pdb", "--receptor_pdb", "mNET.pdb",
     "--poccent_file", "../input/pockets/mNET.pdb_predictions.csv",
     "--name", "A2"], cwd="output", log="output/dock.log")

$ PolyLego-OptimizeBinding --polymers --input_path ./ --ligand_pdb assembly.pdb --receptor_pdb mNET.pdb --poccent_file ../input/pockets/mNET.pdb_predictions.csv --name A2


  PolyLego 1.4.0 | OptimizeBinding
  polymer-protein binding poses
  Finished in 9 s
    docking
  -> exit 0, 16 s


In [6]:
import glob
poses = sorted(glob.glob("output/docking/_*/A2_*_final_0.3.pdb"))
print("\n".join(poses))
show_complex("input/mNET.pdb", ligands=[poses[0]])

output/docking/_0/A2_0_final_0.3.pdb
output/docking/_1/A2_1_final_0.3.pdb
output/docking/_2/A2_2_final_0.3.pdb
output/docking/_3/A2_3_final_0.3.pdb
output/docking/_4/A2_4_final_0.3.pdb


[3D view - run the notebook in Jupyter to display it]


All poses together (one per pocket):

In [7]:
show_complex("input/mNET.pdb", ligands=poses, focus_ligand=False)

[3D view - run the notebook in Jupyter to display it]
